In [20]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-small")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fashion-product-images-small' dataset.
Path to dataset files: /kaggle/input/fashion-product-images-small


In [21]:
import os

# See what's inside
print(os.listdir(path))

['myntradataset', 'images', 'styles.csv']


In [22]:
# Check subfolders
for item in os.listdir(path):
    print(item)

myntradataset
images
styles.csv


In [23]:
import pandas as pd

# Check inside myntradataset folder
print("myntradataset contents:", os.listdir(f"{path}/myntradataset"))

# Load styles.csv
df = pd.read_csv(f"{path}/styles.csv", on_bad_lines='skip')
print("\nShape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

myntradataset contents: ['images', 'styles.csv']

Shape: (44424, 10)

Columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName']

First 5 rows:
      id gender masterCategory subCategory  articleType baseColour  season  \
0  15970    Men        Apparel     Topwear       Shirts  Navy Blue    Fall   
1  39386    Men        Apparel  Bottomwear        Jeans       Blue  Summer   
2  59263  Women    Accessories     Watches      Watches     Silver  Winter   
3  21379    Men        Apparel  Bottomwear  Track Pants      Black    Fall   
4  53759    Men        Apparel     Topwear      Tshirts       Grey  Summer   

     year   usage                             productDisplayName  
0  2011.0  Casual               Turtle Check Men Navy Blue Shirt  
1  2012.0  Casual             Peter England Men Party Blue Jeans  
2  2016.0  Casual                       Titan Women Silver Watch  
3  2011.0  Casual  Manchester United Men 

In [24]:
# See all unique masterCategories (this will be our CNN classes)
print("Master Categories:")
print(df['masterCategory'].value_counts())

print("\nSub Categories:")
print(df['subCategory'].value_counts())

print("\nTotal unique masterCategories:", df['masterCategory'].nunique())

Master Categories:
masterCategory
Apparel           21397
Accessories       11274
Footwear           9219
Personal Care      2403
Free Items          105
Sporting Goods       25
Home                  1
Name: count, dtype: int64

Sub Categories:
subCategory
Topwear                     15402
Shoes                        7343
Bags                         3055
Bottomwear                   2694
Watches                      2542
Innerwear                    1808
Jewellery                    1079
Eyewear                      1073
Fragrance                    1011
Sandal                        963
Wallets                       933
Flip Flops                    913
Belts                         811
Socks                         698
Lips                          527
Dress                         478
Loungewear and Nightwear      470
Saree                         427
Nails                         329
Makeup                        307
Headwear                      293
Ties                         

In [ ]:
import os

# Keep only top 4 categories
categories_to_keep = ['Apparel', 'Accessories', 'Footwear', 'Personal Care']
df_filtered = df[df['masterCategory'].isin(categories_to_keep)]
print("Filtered shape:", df_filtered.shape)
print("\nCategory counts:")
print(df_filtered['masterCategory'].value_counts())

# Check images path
img_folder = f"{path}/myntradataset/images"
sample_images = os.listdir(img_folder)[:5]
print("\nSample image filenames:", sample_images)

# Verify images exist for our data
df_filtered['image_path'] = df_filtered['id'].astype(str) + '.jpg'
existing = df_filtered['image_path'].apply(lambda x: os.path.exists(f"{img_folder}/{x}"))
print("\nImages found:", existing.sum(), "out of", len(df_filtered))

Filtered shape: (44293, 10)

Category counts:
masterCategory
Apparel          21397
Accessories      11274
Footwear          9219
Personal Care     2403
Name: count, dtype: int64

Sample image filenames: ['31973.jpg', '30778.jpg', '19812.jpg', '22735.jpg', '38246.jpg']


/tmp/ipykernel_4000/1100100626.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['image_path'] = df_filtered['id'].astype(str) + '.jpg'


# STEP 2 — Preprocessing

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [ ]:
# Fix the warning from before
df_filtered = df_filtered.copy()
df_filtered['image_path'] = df_filtered['id'].astype(str) + '.jpg'

# Keep only rows where image exists
df_filtered['exists'] = df_filtered['image_path'].apply(
    lambda x: os.path.exists(f"{img_folder}/{x}")
)
df_clean = df_filtered[df_filtered['exists'] == True].reset_index(drop=True)
print("Final dataset size:", len(df_clean))

In [ ]:
# Encode labels (Apparel=0, Accessories=1, Footwear=2, Personal Care=3)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_clean['label'] = le.fit_transform(df_clean['masterCategory'])
print("\nLabel mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {i} → {cls}")

# STEP 3 — Load & Resize Images

In [ ]:
import cv2
from tqdm import tqdm

IMG_SIZE = 64  # resize all images to 64x64
images = []
labels = []

for idx, row in tqdm(df_clean.iterrows(), total=len(df_clean)):
    img_path = f"{img_folder}/{row['image_path']}"
    img = cv2.imread(img_path)
    if img is not None:
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        images.append(img)
        labels.append(row['label'])

# Convert to numpy arrays
images = np.array(images, dtype='float32')
labels = np.array(labels)

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

# STEP 4 — Normalize & Split Data

In [ ]:
from tensorflow.keras.utils import to_categorical

# Normalize pixel values from 0-255 to 0-1
# Makes training faster and more stable
images = images / 255.0

# Convert labels to one-hot encoding
# e.g. label 2 → [0, 0, 1, 0]
labels_encoded = to_categorical(labels, num_classes=4)

# Split into train, validation and test sets
X_train, X_test, y_train, y_test = train_test_split(
    images, labels_encoded, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

#  STEP 5 — Build CNN Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

model = Sequential([
    # Block 1
    Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    # Block 2
    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    # Block 3
    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    # Fully Connected
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')  # 4 categories
])

model.summary()

# STEP 6 — Compile & Train Model

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

# Augmentation
train_datagen = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
)

val_datagen = ImageDataGenerator()

train_generator = train_datagen.flow(X_train, y_train, batch_size=32)
val_generator = val_datagen.flow(X_val, y_val, batch_size=32)

# Early Stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train!
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=[early_stop]
)
# Save model so you don't lose it
model.save('fashion_cnn_model.keras')
print("Model saved!")

# STEP 7 — Evaluate on Test Set

In [ ]:
# Evaluate on unseen test data
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"\nTest Accuracy: {test_accuracy*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# 1. Test Accuracy
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"\nTest Accuracy: {test_accuracy*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

# 2. Predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# 3. Classification Report
class_names = ['Accessories', 'Apparel', 'Footwear', 'Personal Care']
print("\nClassification Report:")
print(classification_report(y_true_classes, y_pred_classes, target_names=class_names))

# 4. Confusion Matrix
cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

# STEP 8 — Training Curves

In [ ]:
plt.figure(figsize=(14,5))

# Accuracy plot
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy', color='blue')
plt.plot(history.history['val_accuracy'], label='Val Accuracy', color='orange')
plt.title('Model Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss plot
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss', color='blue')
plt.plot(history.history['val_loss'], label='Val Loss', color='orange')
plt.title('Model Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# STEP 9 — Visualize Sample Predictions

In [ ]:
# Show sample predictions with actual images
class_names = ['Accessories', 'Apparel', 'Footwear', 'Personal Care']

plt.figure(figsize=(15, 10))
sample_indices = np.random.choice(len(X_test), 12, replace=False)

for i, idx in enumerate(sample_indices):
    plt.subplot(3, 4, i+1)
    plt.imshow(X_test[idx])

    true_label = class_names[y_true_classes[idx]]
    pred_label = class_names[y_pred_classes[idx]]
    confidence = np.max(y_pred[idx]) * 100

    color = 'green' if true_label == pred_label else 'red'
    plt.title(f"True: {true_label}\nPred: {pred_label}\n{confidence:.1f}%",
              color=color, fontsize=8)
    plt.axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Save the model
model.save('fashion_cnn_model.keras')
print("✅ Model saved!")

In [ ]:
# Run this in Colab
from google.colab import files
files.download('fashion_cnn_model.keras')